# Conversational Question Answering (CQA) System using BERT

**Name:** Samar Kh. Saleh


A simplified Conversational Question Answering system built on BERT, extended
with a rule-based dialogue history mechanism to resolve pronouns and
elliptical follow-up questions. The system answers 4 consecutive questions
over the same passage and compares answers generated **with** and **without**
dialogue history.

# Conversational QA System, BERT + Dialogue History

Task: answer 4 questions in a row about the same passage, and show the
difference between answering **with** and **without** memory of earlier
questions.

Model: BERT fine-tuned on SQuAD (extractive QA, it picks the answer
span directly from the passage, doesn't generate free text).

Since BERT normally has no memory between questions, I added a simple
history mechanism: if a question uses a pronoun ("she", "her"...), I
replace it with the actual name from the previous turn before asking BERT.

In [4]:
from transformers import BertTokenizer, BertForQuestionAnswering
import torch
import re

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

### The QA system

`get_answer()` : sends a question + passage to BERT, gets back the answer span.

`resolve_question()`, handles two cases:
1. If the question has a pronoun ("she", "her"...), swap it for the real
   subject name tracked from earlier turns.
2. If the question is very short with no pronoun (like "In what year?"),
   it's probably missing words entirely (ellipsis), merge it with the topic of the last resolved question instead.

`update_subject()` : after each turn, checks if the question named someone
directly, and updates "who we're currently talking about" for next time.

In [5]:
class ConversationalQASystem:
    def __init__(self):
        self.model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
        self.tokenizer = BertTokenizer.from_pretrained(self.model_name)
        self.model = BertForQuestionAnswering.from_pretrained(self.model_name)
        self.model.eval()
        self.current_subject = None
        self.last_topic_question = None  # remembers the last full question

    def resolve_question(self, question):
        if self.current_subject is None:
            return question

        has_pronoun = re.search(r'\b(she|he|they|her|him|them|his|hers|their)\b', question, re.IGNORECASE)

        if has_pronoun:
            resolved = re.sub(r'\b(she|he|they)\b', self.current_subject, question, flags=re.IGNORECASE)
            resolved = re.sub(r'\b(her|him|them)\b', self.current_subject, resolved, flags=re.IGNORECASE)
            resolved = re.sub(r'\b(her|his|their)\b', f"{self.current_subject}'s", resolved, flags=re.IGNORECASE)
            self.last_topic_question = resolved
            return resolved

        # elliptical question (very short, no pronoun) -> merge with last topic
        if len(question.split()) <= 4 and self.last_topic_question:
            # e.g. "In what year?" + "What prize did Irène win?"
            # -> "In what year did Irène win?"
            merged = question.rstrip("?") + " " + re.sub(
                r'^(what|who|where|when|which)\s+\w+\s+', '', self.last_topic_question, flags=re.IGNORECASE
            )
            self.last_topic_question = merged
            return merged

        self.last_topic_question = question
        return question

    def update_subject(self, question, known_entities):
        """If the question names someone explicitly, update the current subject."""
        for e in known_entities:
            if e in question:
                self.current_subject = e
                break

    def get_answer(self, question, context, use_history=True):
        query = self.resolve_question(question) if use_history else question

        inputs = self.tokenizer(query, context, add_special_tokens=True, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model(**inputs)

        start_scores = outputs.start_logits[0]
        end_scores = outputs.end_logits[0]

        token_type_ids = inputs["token_type_ids"][0]
        context_mask = token_type_ids == 1
        start_scores = start_scores.masked_fill(~context_mask, float('-inf'))
        end_scores = end_scores.masked_fill(~context_mask, float('-inf'))

        answer_start = torch.argmax(start_scores)
        answer_end = torch.argmax(end_scores) + 1

        input_ids = inputs["input_ids"][0]
        answer = self.tokenizer.convert_tokens_to_string(
            self.tokenizer.convert_ids_to_tokens(input_ids[answer_start:answer_end])
        )
        return answer.strip()

### Passage + conversation

Passage mentions **two people** (Marie Curie and her daughter Irène) on
purpose, this makes "she" genuinely ambiguous, so we can actually test
if history helps.

In [6]:
passage = """
Marie Curie and her daughter Irène Joliot-Curie were both pioneering scientists.
Marie was born in Warsaw in 1867 and began her research in Paris in 1891.
Irène was born in Paris in 1897 and began her own research in 1918.
Marie won her first Nobel Prize in Physics in 1903.
Irène later won the Nobel Prize in Chemistry in 1935.
"""

known_entities = ["Marie Curie", "Irène Joliot-Curie", "Irène", "Marie"]

questions = [
    "Who was Irène Joliot-Curie?",
    "What year did she begin her research?",
    "What prize did she win?",
    "In what year?"
]

cqa_system = ConversationalQASystem()

print(f"Passage: {passage}\n" + "-"*50)
for q in questions:
    ans_with = cqa_system.get_answer(q, passage, use_history=True)
    ans_without = cqa_system.get_answer(q, passage, use_history=False)
    print(f"User: {q}")
    print(f"System (With History): {ans_with}")
    print(f"System (Without History): {ans_without}\n")
    cqa_system.update_subject(q, known_entities)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Passage: 
Marie Curie and her daughter Irène Joliot-Curie were both pioneering scientists.
Marie was born in Warsaw in 1867 and began her research in Paris in 1891.
Irène was born in Paris in 1897 and began her own research in 1918.
Marie won her first Nobel Prize in Physics in 1903.
Irène later won the Nobel Prize in Chemistry in 1935.

--------------------------------------------------
User: Who was Irène Joliot-Curie?
System (With History): pioneering scientists
System (Without History): pioneering scientists

User: What year did she begin her research?
System (With History): 1918
System (Without History): 1891

User: What prize did she win?
System (With History): nobel prize in chemistry
System (Without History): nobel prize in physics

User: In what year?
System (With History): 1935
System (Without History): 



### What happened, turn by turn

- **Turn 1** : same answer both ways. Question already names Irène
  directly, nothing to resolve.
- **Turn 2** : without history: "1891" (Marie's year, wrong person).
  With history: "she" gets replaced with "Irène" -> correctly returns 1918.
- **Turn 3** : without history: "nobel prize in physics" (Marie's prize,
  wrong person). With history: correctly returns "nobel prize in chemistry".
- **Turn 4** ("In what year?") : this one has no pronoun at all, it's
  missing the subject and verb ("in what year did she win it?").
  Without history: model finds nothing, returns empty. With history:
  the ellipsis-handling merges it with the previous resolved question
  ("what prize did Irène win"), producing something like "in what year
  did Irène win", and the model correctly returns 1935.

## Report

#### **Model:**
BERT fine-tuned on SQuAD (`bert-large-uncased-whole-word-masking-finetuned-squad`),
an extractive QA model, it selects the answer span directly from the
passage rather than generating text. BERT alone has no memory across
turns, so I added a dialogue-history mechanism with two parts:
1. pronoun resolution, track who the conversation is about and substitute pronouns with the real name.
2. ellipsis handling, if a question is very short with no pronoun, merge it with the topic of the previous resolved question.

#### **Setup:**
One passage mentioning two people (Marie Curie and her
daughter Irène), with 4 connected questions, where later ones use "she"
or omit the subject entirely, referring back to Irène.

#### **Results:**
Dialogue history changed the answer in 3 out of 4 turns.
Without history, the system consistently defaulted to answering about
Marie (likely because she's mentioned first/more prominently in the
passage), giving wrong answers for turns 2, 3, and 4. With history, all
three were correctly resolved to Irène's facts: research year (1918),
prize (Chemistry), and prize year (1935). Turn 4 is the most interesting
case, it has no pronoun to resolve, so a simple pronoun-swap approach
alone would have failed here (as it did in without-history condition,
returning an empty answer); merging it with the previous question's topic
was needed to recover the missing subject and verb.

#### **Conclusion:**
This shows dialogue history is not just about resolving
pronouns, real conversations often drop words entirely from follow-up
questions (ellipsis), and a working CQA system needs to handle both. This
is a simplified, rule-based approximation of what models like BiDAF++ and
FlowQA do automatically through learned attention over dialogue history,
rather than explicit text rewriting rules.